# 04 · Databricks reads: tables, queries, files

**Run first:**
```bash
uv run pytest -v tests/unit/dtrack_right/backends/test_targets.py
just test spark
DTRACK_LIVE_KEEP=1 just live test dtrack_right
aws s3 ls --recursive s3://dve-tools/dtrack/work/inputs/
```
**Read:** `src/dtrack_right/backends/targets.py::SparkTarget`.

A fake Spark session records every statement, so we can read the exact SQL the job sends.

In [ ]:
import nbkit
import pandas as pd
from nbkit import code, table

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

In [ ]:
from dtrack_right.backends.targets import SparkTarget

FakeSpark, FakeRow = nbkit.support.FakeSpark, nbkit.support.FakeRow
configs = {
    "table": {"catalog": "dve_catlog", "schema": "dve_live", "table": "orders"},
    "legacy macro": {"conn_macro": "dve_catlog.dve_live", "table": "orders"},
    "query": {
        "kind": "query",
        "query": "SELECT * FROM dve_catlog.dve_live.orders WHERE region <> 'XX'",
    },
    "files + patch": {
        "kind": "files",
        "path": "/Volumes/dve_catlog/dve_live/work/inputs/orders/",
        "format": "csv",
        "options": {"header": "true"},
        "schema_hints": "amount DECIMAL(18,2)",
        "query": "SELECT *, to_date(d_str, 'dd/MM/yyyy') AS business_date FROM {src}",
    },
}
table(
    pd.DataFrame(
        [
            {"config": name, "relation()": SparkTarget(None, config).relation()}
            for name, config in configs.items()
        ]
    )
)

## 1 · Preflight uses `DESCRIBE QUERY`

`DESCRIBE <name>` accepts only a table or view name. A subquery or `read_files(...)` is not a
name, so preflight wraps any relation in a query and asks Spark for the output schema.

In [ ]:
describe = [
    ("business_date", "string", None),
    ("order_id", "bigint", None),
    ("amount", "decimal(10,2)", None),
    ("status", "string", None),
]
spark = FakeSpark(lambda sql: describe)
target = SparkTarget(spark, {**configs["files + patch"], "date_col": "business_date"})
print(target.preflight())
code(spark.statements[0], "sql", "preflight statement")

## 2 · Deterministic sampling, not `TABLESAMPLE`

Both sides must pick the **same keys** independently. Each side normalises the key, hashes it
with MD5, keeps the first 8 hex digits as an integer, and compares with a threshold. The Python
rule (`selected_fraction`) and the Spark SQL are the same formula.

In [ ]:
import hashlib

from dtrack_right.common.hashing import hash32, normalize_value, selected_fraction

threshold = round(0.25 * 2**32)
keys = [str(100000 + n) for n in range(2000)]
picked = [k for k in keys if selected_fraction([k], ["numeric"], threshold)]
print(f"picked {len(picked)} of {len(keys)} (target 25%)")
key = normalize_value("100042.0", "numeric")
print(key, hash32(key) == int(hashlib.md5(key.encode()).hexdigest()[:8], 16))
manifest = {
    "sample": {"mode": "fraction", "threshold": threshold},
    "right_key_cols": ["order_id"],
    "key_families": ["numeric"],
}
code(
    SparkTarget(None, {"table": "t"})._sample_predicate(manifest, {}),
    "sql",
    "Right-side sampling predicate",
)

## 3 · Column types must agree on both sides

The Left side decides numeric from the **values** (`pd.to_numeric`); the Right side from the
**Spark type**. If the Right side reads a file as all strings (`inferColumnTypes=false`), the
same column becomes numeric on the left and categorical on the right.

**Common mistake:** "it still matches, counts are equal". Counts are equal, but `hash_sum`
(decimal collapsing only for numeric), `mean`, and `min`/`max` (numeric vs lexical order) all
differ, so the verdict is `diff`. `col_type_overrides` fixes it by deciding the type per pair.

In [ ]:
from dtrack_right.checks.col_check import compare_statistics
from dtrack_right.common.stats import column_statistics

amount = pd.read_csv(nbkit.ROOT / "examples/e2e/data/left/ORDERS.csv", dtype=object)["AMOUNT"].head(
    200
)
rows = []
for label, right_type in (("no override", "categorical"), ("override numeric", "numeric")):
    left_stats, right_stats = (
        column_statistics(amount),
        column_statistics(amount, col_type=right_type),
    )
    result = compare_statistics(
        pair="orders", window="w", column="AMOUNT", left=left_stats, right=right_stats
    )
    rows.append(
        {
            "case": label,
            "verdict": result["verdict"],
            "hash_sum_match": result["hash_sum_match"],
            "min_left": result["min_left"],
            "min_right": result["min_right"],
            "max_right": result["max_right"],
        }
    )
table(pd.DataFrame(rows).astype({"hash_sum_match": str}))

## 4 · Where the bytes live

| Object | Storage | Seen in `aws s3 ls s3://dve-tools/...`? |
| --- | --- | --- |
| Volume `work`, `report` (external) | `s3://dve-tools/dtrack/{work,report}` | yes, same objects under two names |
| files uploaded to `/Volumes/.../work/inputs/` | the same S3 prefix | yes |
| managed tables (`orders_live_...`) | the workspace's managed storage (`s3://dbstorage-prod-...`, Databricks' account) | no |